In [2]:
import numpy as np
import pandas as pd

# Task 0
Read the dataset from csv file & perform data cleaning - remove all rows, which contains `?` in some columns.
Also check for data correctness (salary & salary $K).

In [3]:
data = pd.read_csv("../data/adult.csv", index_col=0, na_values="?").dropna()

salary_is_correct = data["salary"].eq(">50K").eq(
    data["salary K$"].gt(50)
)

print(f"Rows after cleaning: {data.shape[0]}")
print(f"Salary columns are consistent: {salary_is_correct.all()}")
data.head()

Rows after cleaning: 30162
Salary columns are consistent: True


,age,workclass,education,marital-status,occupation,relationship,race,sex,hours-per-week,native-country,salary,salary K$
0,39,State-gov,Bachelors,Never-married,Adm-clerical,Not-in-family,White,Male,40,United-States,<=50K,39
1,50,Self-emp-not-inc,Bachelors,Married-civ-spouse,Exec-managerial,Husband,White,Male,13,United-States,<=50K,35
2,38,Private,HS-grad,Divorced,Handlers-cleaners,Not-in-family,White,Male,40,United-States,<=50K,27
3,53,Private,11th,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,40,United-States,<=50K,43
4,28,Private,Bachelors,Married-civ-spouse,Prof-specialty,Wife,Black,Female,40,Cuba,<=50K,25


# Task 1
Print the count of men and women in the dataset.

In [5]:
data["sex"].value_counts()

sex
Male      20380
Female     9782
Name: count, dtype: int64

# Task 2
Find the average age of men in dataset

In [6]:
average_male_age = data.loc[data["sex"].eq("Male"), "age"].mean()
print(f"Average age of men: {average_male_age:.2f}")

Average age of men: 39.18


# Task 3
Get the percentage of people from Poland (native-country)

In [7]:
poland_percentage = data["native-country"].eq("Poland").mean() * 100
print(f"Percentage of people from Poland: {poland_percentage:.2f}%")

Percentage of people from Poland: 0.19%


# Task 4
Get the mean and standard deviation of the age for people who earn > 50K per year. After this, get it for those who earn <= 50K.

In [8]:
age_statistics_by_salary = (
    data.groupby("salary")["age"].describe().loc[:, "mean":"std"]
)
age_statistics_by_salary

,mean,std
salary,,
<=50K,36.60806,13.464631
>50K,43.95911,10.269633


# Task 5
Check, if there are some people without higher education (education: Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters, Doctorate), but with > 50K salary

In [9]:
has_higher_education = data["education"].str.fullmatch(
    r"Bachelors|Prof-school|Assoc-acdm|Assoc-voc|Masters|Doctorate"
)
high_income_without_higher_education = (
    ~has_higher_education & data["salary"].eq(">50K")
)

print(f"Such people exist: {high_income_without_higher_education.any()}")
print(f"Their count: {high_income_without_higher_education.sum()}")

Such people exist: True
Their count: 3178


# Task 6
Get the statistics of age for each type of education. Use `groupby` and `describe` for this.

In [10]:
education_age_statistics = data.groupby("education")["age"].describe()
education_age_statistics

,count,mean,std,min,25%,50%,75%,max
education,,,,,,,,
10th,820.0,37.897561,16.225795,17.0,23.0,36.0,52.0,90.0
11th,1048.0,32.363550,15.089307,17.0,18.0,28.5,43.0,90.0
12th,377.0,32.013263,14.373710,17.0,19.0,28.0,41.0,79.0
1st-4th,151.0,44.622517,14.929051,19.0,33.0,44.0,56.0,81.0
5th-6th,288.0,41.649306,14.754622,17.0,28.0,41.0,53.0,82.0
7th-8th,557.0,47.631957,15.737479,17.0,34.0,49.0,60.0,90.0
9th,455.0,40.303297,15.335754,17.0,28.0,38.0,53.0,90.0
Assoc-acdm,1008.0,37.286706,10.509755,19.0,29.0,36.0,44.0,90.0
Assoc-voc,1307.0,38.246366,11.181253,19.0,30.0,37.0,45.0,84.0


# Task 7
Compare the married and non-married men salaries. Who earns more? (>50K or <=50K)
Married men are those, whom `marital-status` starts with "Married". Others are not.

In [12]:
men = data.loc[data["sex"].eq("Male")]
is_married = (
    men["marital-status"].str.startswith("Married").rename("is_married")
)
salary_counts = pd.crosstab(is_married, men["salary"])
salary_percentages = (
    salary_counts.div(salary_counts.sum(axis=1), axis=0).mul(100)
)

print("Counts:")
print(salary_counts)
print("\nPercentages within each group:")
print(salary_percentages)
print(
    "\nIn both groups most men earn <=50K, but married men are much "
    "more likely to earn >50K than non-married men."
)

Counts:
salary      <=50K  >50K
is_married             
False        6932   673
True         7052  5723

Percentages within each group:
salary          <=50K       >50K
is_married                      
False       91.150559   8.849441
True        55.201566  44.798434

In both groups most men earn <=50K, but married men are much more likely to earn >50K than non-married men.


# Task 8
Get the max hours per week some person works. How many people works the same amount of hours per week?

In [13]:
max_hours_per_week = data["hours-per-week"].max()
people_with_max_hours = data["hours-per-week"].eq(max_hours_per_week).sum()

print(f"Maximum hours per week: {max_hours_per_week}")
print(f"Number of people who work that many hours: {people_with_max_hours}")

Maximum hours per week: 99
Number of people who work that many hours: 78


# Task 9
Analyze the correlation between data in dataset. Understand connected fields in it and print highlight thier connection.

In [14]:
correlation = data.select_dtypes(include=np.number).corr()
upper_triangle = np.triu(np.ones(correlation.shape, dtype=bool), k=1)
correlation_pairs = (
    correlation.where(upper_triangle).stack().sort_values(key=np.abs, ascending=False)
)
strongest_pair = correlation_pairs.index[0]
strongest_value = correlation_pairs.iloc[0]

print(correlation)
print(
    f"\nStrongest numeric connection: {strongest_pair[0]} and "
    f"{strongest_pair[1]} (correlation = {strongest_value:.3f}). "
    "It is a weak positive correlation."
)

                     age  hours-per-week  salary K$
age             1.000000        0.101599   0.208203
hours-per-week  0.101599        1.000000   0.196378
salary K$       0.208203        0.196378   1.000000

Strongest numeric connection: age and salary K$ (correlation = 0.208). It is a weak positive correlation.
